# Register the NeMo TFM Pipeline

Run this notebook **once** from the workbench to compile the pipeline definition
and upload it to the OpenShift AI pipeline server.

After running, you will be able to:
- See the pipeline in the **Pipelines** tab of the RHOAI dashboard
- Create and trigger runs directly from the dashboard UI
- Track each notebook step's logs, inputs, and outputs

---

**Pre-requisite:** the project notebooks must be present at `/opt/app-root/src/`  
If this is a fresh workbench, clone the repo first:
```bash
git clone https://github.com/robbybrodie/transaction-foundation-model-openshiftai.git /opt/app-root/src/nemo-tfm
```

## 1 · Install KFP SDK

`kfp` and `kfp-kubernetes` are not in the base image — install them here.  
This takes ~30 seconds and does not affect any other notebooks.

In [ ]:
%pip install --quiet 'kfp>=2.7,<3' 'kfp-kubernetes>=1.3,<2'

## 2 · Compile the pipeline

Turns the Python pipeline definition into a YAML spec that KFP understands.

In [ ]:
import sys, os

# Make sure the pipeline module is importable whether we're in /opt/app-root/src
# or in /opt/app-root/src/nemo-tfm (cloned repo subdirectory)
for candidate in [
    "/opt/app-root/src",
    "/opt/app-root/src/nemo-tfm",
    "/opt/app-root/src/transaction-foundation-model-openshiftai",
]:
    if os.path.isfile(os.path.join(candidate, "pipeline", "nemo_tfm_pipeline.py")):
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find pipeline/nemo_tfm_pipeline.py — "
        "clone the repo to /opt/app-root/src first."
    )

sys.path.insert(0, REPO_ROOT)
print(f"Using repo root: {REPO_ROOT}")

from kfp import compiler
from pipeline.nemo_tfm_pipeline import nemo_tfm_pipeline, PIPELINE_NAME

PIPELINE_YAML = os.path.join(REPO_ROOT, "pipeline", "nemo_tfm_pipeline.yaml")
compiler.Compiler().compile(nemo_tfm_pipeline, PIPELINE_YAML)
print(f"Compiled → {PIPELINE_YAML}")

## 3 · Connect to the pipeline server

In [ ]:
import kfp

# In-cluster endpoint — reachable from any workbench pod without going via the Route
KFP_ENDPOINT = "https://ds-pipeline-pipelines-definition.nemo-tfm.svc.cluster.local:8443"
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"

with open(SA_TOKEN_PATH) as f:
    token = f.read().strip()

client = kfp.Client(
    host=KFP_ENDPOINT,
    existing_token=token,
    verify_ssl=False,   # cluster uses internal self-signed cert
)

# Smoke-test — list existing pipelines
existing = client.list_pipelines()
print(f"Connected. Pipelines already registered: {existing.total_size}")

## 4 · Upload (or update) the pipeline

In [ ]:
# Check whether the pipeline already exists so we can version it cleanly
existing_ids = {
    p.display_name: p.pipeline_id
    for p in (existing.pipelines or [])
}

if PIPELINE_NAME in existing_ids:
    pipeline_id = existing_ids[PIPELINE_NAME]
    result = client.upload_pipeline_version(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_version_name="latest",
        pipeline_id=pipeline_id,
    )
    print(f"Updated existing pipeline '{PIPELINE_NAME}' (id={pipeline_id})")
    print(f"New version id: {result.pipeline_version_id}")
else:
    result = client.upload_pipeline(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_name=PIPELINE_NAME,
    )
    print(f"Registered new pipeline '{PIPELINE_NAME}'")
    print(f"Pipeline id: {result.pipeline_id}")

print("\nDone — refresh the Pipelines tab in the RHOAI dashboard to see it.")

---
## What happens when you run the pipeline?

| Step | Notebook | What it does |
|------|----------|--------------|
| 1 | `01_dataset_baseline.ipynb` | Load TabFormer dataset, temporal splits, XGBoost baseline |
| 2 | `02_seq_preproc_tokenization.ipynb` | GPU-accelerated tokeniser pipeline (cuDF) |
| 3 | `03_foundation_model_training.ipynb` | Pre-train NeMo decoder (30-step demo → full run) |
| 4 | `04_inference_embedding_extraction.ipynb` | Extract 512-d embeddings, UMAP visualisation |
| 5 | `05_xgboost_fraud_detection.ipynb` | Compare XGBoost with raw features vs. embeddings |

Each step's executed notebook is saved to `pipeline-outputs/` on the shared PVC,  
so you can inspect every cell's output after the run — even if you didn't watch it live.